# FE and improving


In [29]:
import logging
import os

log_path = r'/Users/sanjarbekkakhramonov/Desktop/Python course/5-oy/Project/logs/improving.log'

os.makedirs(os.path.dirname(log_path), exist_ok=True)

logging.basicConfig(
    filename=log_path,
    filemode='a',
    level=logging.INFO,
    format="%(asctime)s-%(levelname)s-%(message)s"
)

logging.info("Improving boshlandi")


In [30]:
import pandas as pd
try:
    df = pd.read_csv("/Users/sanjarbekkakhramonov/Desktop/Python course/5-oy/Project/data/raw/FR_raw.csv")
    logging.info(f"✅ Dataset yuklandi: {df.shape}")


except Exception as e:
    logging.exception("🔥 Dataset yuklashda kutilmagan xato!")
    raise




df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5320 entries, 0 to 5319
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   season        5320 non-null   object
 1   home_team     5320 non-null   object
 2   away_team     5320 non-null   object
 3   home_goals    5320 non-null   int64 
 4   away_goals    5320 non-null   int64 
 5   target        5320 non-null   int64 
 6   season_start  5320 non-null   int64 
dtypes: int64(4), object(3)
memory usage: 291.1+ KB


In [31]:
team_map = {
    "Arsenal": "ARS",
    "Aston Villa": "AVL",
    "Birmingham City": "BIR",
    "Blackburn Rovers": "BLB",
    "Blackpool": "BLP",
    "Bolton Wanderers": "BOL",
    "Bournemouth": "BOU",
    "Brentford": "BRE",
    "Brighton & Hove Albion": "BHA",
    "Burnley": "BUR",
    "Cardiff City": "CAR",
    "Chelsea": "CHE",
    "Crystal Palace": "CRY",
    "Everton": "EVE",
    "Fulham": "FUL",
    "Huddersfield Town": "HUD",
    "Hull City": "HUL",
    "Leeds United": "LEE",
    "Leicester City": "LEI",
    "Liverpool": "LIV",
    "Luton Town": "LUT",
    "Manchester City": "MCI",
    "Manchester United": "MUN",
    "Middlesbrough": "MID",
    "Newcastle United": "NEW",
    "Norwich City": "NOR",
    "Nottingham Forest": "NFO",
    "Queens Park Rangers": "QPR",
    "Reading": "REA",
    "Sheffield United": "SHU",
    "Southampton": "SOU",
    "Stoke City": "STK",
    "Sunderland": "SUN",
    "Swansea City": "SWA",
    "Tottenham Hotspur": "TOT",
    "Watford": "WAT",
    "West Bromwich Albion": "WBA",
    "West Ham United": "WHU",
    "Wigan Athletic": "WIG",
    "Wolverhampton Wanderers": "WOL"
}

In [32]:
# 1) raw loaddan keyin darrov
df["home_team"] = df["home_team"].map(team_map)

# 2) tekshiruv a
print(df["home_team"].head(5).tolist())
print("NaN:", df["home_team"].isna().sum())

['ARS', 'ARS', 'ARS', 'ARS', 'ARS']
NaN: 0


In [33]:

# seasons = sorted(df["season"].unique())

# # oxirgi 3 tasi
# test_seasons = seasons[-3:]

# train = df[~df["season"].isin(test_seasons)].copy()
# test  = df[df["season"].isin(test_seasons)].copy()

# print("Train seasons:", sorted(train["season"].unique()))
# print("Test seasons:", sorted(test["season"].unique()))

In [34]:
from collections import deque, defaultdict
import numpy as np

def add_last10_overall(df):
    pts_hist = defaultdict(lambda: deque(maxlen=10))
    gd_hist  = defaultdict(lambda: deque(maxlen=10))

    home_last10_pts = np.full(len(df), np.nan)
    away_last10_pts = np.full(len(df), np.nan)
    home_last10_gd  = np.full(len(df), np.nan)
    away_last10_gd  = np.full(len(df), np.nan)

    for i, row in enumerate(df.itertuples(index=False)):
        ht = row.home_team
        at = row.away_team

        # BEFORE match → oxirgi 10 o'yin yig'indisi
        home_last10_pts[i] = sum(pts_hist[ht]) if pts_hist[ht] else np.nan
        away_last10_pts[i] = sum(pts_hist[at]) if pts_hist[at] else np.nan

        home_last10_gd[i]  = sum(gd_hist[ht]) if gd_hist[ht] else np.nan
        away_last10_gd[i]  = sum(gd_hist[at]) if gd_hist[at] else np.nan

        # AFTER match → history update
        if row.target == 2:      # home win
            hp, ap = 3, 0
        elif row.target == 1:    # draw
            hp, ap = 1, 1
        else:                    # away win
            hp, ap = 0, 3

        hgd = row.home_goals - row.away_goals
        agd = -hgd

        pts_hist[ht].append(hp)
        pts_hist[at].append(ap)
        gd_hist[ht].append(hgd)
        gd_hist[at].append(agd)

    out = df.copy()
    out["home_last10_points"] = home_last10_pts
    out["away_last10_points"] = away_last10_pts
    out["home_last10_gd"] = home_last10_gd
    out["away_last10_gd"] = away_last10_gd


    return out

In [35]:
df = add_last10_overall(df)

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5320 entries, 0 to 5319
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   season              5320 non-null   object 
 1   home_team           5320 non-null   object 
 2   away_team           5320 non-null   object 
 3   home_goals          5320 non-null   int64  
 4   away_goals          5320 non-null   int64  
 5   target              5320 non-null   int64  
 6   season_start        5320 non-null   int64  
 7   home_last10_points  5319 non-null   float64
 8   away_last10_points  5281 non-null   float64
 9   home_last10_gd      5319 non-null   float64
 10  away_last10_gd      5281 non-null   float64
dtypes: float64(4), int64(4), object(3)
memory usage: 457.3+ KB


In [37]:
from collections import deque, defaultdict
import numpy as np

def add_h2h_last5_points(df, k=5):
    # pair -> last k matches: (home, away, target)
    h2h = defaultdict(lambda: deque(maxlen=k))

    home_pts = np.full(len(df), np.nan)
    away_pts = np.full(len(df), np.nan)

    for i, row in enumerate(df.itertuples(index=False)):
        ht, at = row.home_team, row.away_team
        pair = tuple(sorted((ht, at)))

        past = h2h[pair]
        if past:
            h_sum = a_sum = 0

            for p_home, p_away, p_target in past:
                # points in that past match from past-home perspective
                if p_target == 2:
                    ph, pa = 3, 0
                elif p_target == 1:
                    ph, pa = 1, 1
                else:
                    ph, pa = 0, 3

                # map to CURRENT perspective
                if ht == p_home:   # current home was past home
                    h_sum += ph
                    a_sum += pa
                else:              # current home was past away
                    h_sum += pa
                    a_sum += ph

            home_pts[i] = h_sum
            away_pts[i] = a_sum

        # update history AFTER current match
        h2h[pair].append((ht, at, row.target))

    out = df.copy()
    out[f"h2h_last{k}_home_points"] = home_pts
    out[f"h2h_last{k}_away_points"] = away_pts
    return out

In [38]:
df = add_h2h_last5_points(df, k=5)

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5320 entries, 0 to 5319
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   season                 5320 non-null   object 
 1   home_team              5320 non-null   object 
 2   away_team              5320 non-null   object 
 3   home_goals             5320 non-null   int64  
 4   away_goals             5320 non-null   int64  
 5   target                 5320 non-null   int64  
 6   season_start           5320 non-null   int64  
 7   home_last10_points     5319 non-null   float64
 8   away_last10_points     5281 non-null   float64
 9   home_last10_gd         5319 non-null   float64
 10  away_last10_gd         5281 non-null   float64
 11  h2h_last5_home_points  4716 non-null   float64
 12  h2h_last5_away_points  4716 non-null   float64
dtypes: float64(6), int64(4), object(3)
memory usage: 540.4+ KB


In [40]:
df.drop(columns=["home_goals", "away_goals","season"], inplace=True)
df.dropna( inplace=True)

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4716 entries, 19 to 5319
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   home_team              4716 non-null   object 
 1   away_team              4716 non-null   object 
 2   target                 4716 non-null   int64  
 3   season_start           4716 non-null   int64  
 4   home_last10_points     4716 non-null   float64
 5   away_last10_points     4716 non-null   float64
 6   home_last10_gd         4716 non-null   float64
 7   away_last10_gd         4716 non-null   float64
 8   h2h_last5_home_points  4716 non-null   float64
 9   h2h_last5_away_points  4716 non-null   float64
dtypes: float64(6), int64(2), object(2)
memory usage: 405.3+ KB


In [42]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import numpy as np

# 1) split
seasons = sorted(df["season_start"].unique())
test_seasons = seasons[-3:]

train = df[~df["season_start"].isin(test_seasons)].copy()
test  = df[df["season_start"].isin(test_seasons)].copy()

X_train = train.drop(columns=["target"])
y_train = train["target"]
X_test  = test.drop(columns=["target"])
y_test  = test["target"]

cat_cols = ["home_team","away_team"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

# 2) encoding (fit only on train)
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train[cat_cols] = enc.fit_transform(X_train[cat_cols])
X_test[cat_cols]  = enc.transform(X_test[cat_cols])

# 3) scaling (faqat kerak bo'lsa, masalan LogReg uchun)
# Tree model (RF) uchun scale shart emas.
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

# 4) endi model train qilasan va feature importance olasan

In [43]:
import os
try:
    improved_path = "/Users/sanjarbekkakhramonov/Desktop/Python course/5-oy/Project/data/improved"
    os.makedirs(improved_path, exist_ok=True)

    X_train.to_csv(f"{improved_path}/X_train_improved.csv", index=False)
    X_test.to_csv(f"{improved_path}/X_test_improved.csv", index=False)
    y_train.to_csv(f"{improved_path}/y_train_improved.csv", index=False)
    y_test.to_csv(f"{improved_path}/y_test_improved.csv", index=False)


    logging.info("Improved datasetlar saqlandi")
except Exception:
    logging.exception("Improved dataset saqlashda kutilmagan xato!")
    raise